# Capítulo 5 · Árboles: clasificar y predecir un número

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/lectures/chapter_05/05_arboles_clasificacion_y_regresion.ipynb)

**Aprendizaje supervisado · Uso del árbol de decisión**

Un árbol de decisión sirve para las dos tareas. La pregunta que hace es la misma (`¿esta variable es menor que este umbral?`). Cambia lo que contesta la hoja:

- **Clasificación:** la clase más frecuente de los ejemplos que cayeron ahí.
- **Regresión:** el promedio del número que queremos predecir.

Si no tiene límite, sigue partiendo hasta memorizar el entrenamiento. `max_depth` corta ese crecimiento.

### 📚 Índice
1. Iris: un vistazo y clasificar
2. Hasta dónde dejar crecer el árbol
3. Autos: un vistazo y predecir millas por galón
4. Chuleta resumen


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

sns.set_theme(style="whitegrid")
palette = sns.color_palette("deep")

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = raiz / "data"


---
## 1. Iris: un vistazo y clasificar

150 flores, 4 medidas en centímetros y 3 especies. `target` vale 0, 1 o 2: setosa, versicolor y virginica.


In [ ]:
iris = pd.read_csv(DATA_DIR / "iris.csv")
especies = ["setosa", "versicolor", "virginica"]
iris.head()


### Cómo se ve

Tres especies con 50 flores cada una. En el pétalo, la setosa queda aparte; versicolor y virginica se acercan. El largo y el ancho del pétalo van casi juntos.


In [ ]:
iris["target"].map(dict(enumerate(especies))).value_counts().rename("flores")


In [ ]:
iris.drop(columns="target").describe()


In [ ]:
iris_plot = iris.assign(especie=iris["target"].map(dict(enumerate(especies))))
sns.scatterplot(
    data=iris_plot, x="petal length (cm)", y="petal width (cm)",
    hue="especie", palette=palette[:3]
)
plt.show()


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    iris.drop(columns="target").corr(), annot=True, fmt=".2f",
    cmap="vlag", center=0, vmin=-1, vmax=1
)
plt.show()


`stratify=y` mantiene la misma proporción de especies en train y en test. En regresión no se usa.


In [ ]:
X = iris.drop(columns="target")
y = iris["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

arbol = DecisionTreeClassifier(random_state=42)
arbol.fit(X_train, y_train)

print(f"Accuracy en train: {arbol.score(X_train, y_train):.2f}")
print(f"Accuracy en test:  {arbol.score(X_test, y_test):.2f}")
print(f"Hojas: {arbol.get_n_leaves()}")


El 1.00 en train no es una buena noticia: el árbol siguió preguntando hasta dejar cada hoja pura. Abajo se dibujan solo las dos primeras preguntas, que ya separan a la setosa.


In [ ]:
plt.figure(figsize=(12, 6))
plot_tree(
    arbol, max_depth=2, filled=True, feature_names=X.columns, class_names=especies,
    fontsize=9
)
plt.show()


---
## 2. Hasta dónde dejar crecer el árbol

`max_depth` es el número máximo de preguntas seguidas. Sin ese argumento, scikit-learn no pone límite.

> 💡 `min_samples_leaf=5` es la otra forma habitual de frenarlo: ninguna hoja se queda con menos de 5 flores.


In [ ]:
filas = []
for profundidad in [1, 2, 3, None]:
    modelo = DecisionTreeClassifier(max_depth=profundidad, random_state=42)
    modelo.fit(X_train, y_train)
    filas.append({
        "max_depth": "sin límite" if profundidad is None else profundidad,
        "accuracy_train": modelo.score(X_train, y_train),
        "accuracy_test": modelo.score(X_test, y_test),
    })

pd.DataFrame(filas).round(2)


---
## 3. Autos: un vistazo y predecir las millas por galón

El regresor se usa igual: `fit` y `predict`. En la hoja no hay una clase, hay el **promedio** de `mpg` de los autos que cayeron ahí. Por eso la predicción es una escalera, no una recta.


### Cómo se ve

398 autos. `horsepower` es la única columna con huecos: 6 filas. El peso y las millas por galón se mueven juntos, al revés: a más peso, menos `mpg`.


In [ ]:
autos = pd.read_csv(DATA_DIR / "mpg.csv")
autos.head()


In [ ]:
autos.isna().sum().rename("nulos")


In [ ]:
autos.describe()


In [ ]:
sns.histplot(data=autos, x="mpg", bins=20, color=palette[0])
plt.show()


In [ ]:
sns.scatterplot(data=autos, x="weight", y="mpg", color=palette[0], alpha=0.7)
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(autos.corr(numeric_only=True), annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1)
plt.show()


In [ ]:
autos = autos.dropna()
X_peso = autos[["weight"]]
y_mpg = autos["mpg"]
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_peso, y_mpg, test_size=0.2, random_state=42
)

regresor = DecisionTreeRegressor(max_depth=2, random_state=42)
regresor.fit(X_train_p, y_train_p)

pesos = np.linspace(autos["weight"].min(), autos["weight"].max(), 300)
prediccion = regresor.predict(pd.DataFrame({"weight": pesos}))

sns.scatterplot(data=autos, x="weight", y="mpg", color=palette[0], alpha=0.6)
plt.plot(pesos, prediccion, color=palette[1], linewidth=2)
plt.title("Árbol de profundidad 2: cuatro promedios")
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
plot_tree(regresor, filled=True, feature_names=["weight"], fontsize=9)
plt.show()


Con más variables el uso no cambia. Aquí entran peso, potencia y año. Sin límite, el $R^2$ de train llega casi a 1 y se separa del de test: el árbol ajusta autos sueltos del entrenamiento. Con profundidad 4 el de test queda parecido y el de train ya no está pegado a 1.


In [ ]:
columnas = ["weight", "horsepower", "model_year"]
X_auto = autos[columnas]
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_auto, y_mpg, test_size=0.2, random_state=42
)

filas = []
for profundidad in [2, 4, None]:
    modelo = DecisionTreeRegressor(max_depth=profundidad, random_state=42)
    modelo.fit(X_train_a, y_train_a)
    pred_test = modelo.predict(X_test_a)
    filas.append({
        "max_depth": "sin límite" if profundidad is None else profundidad,
        "R2_train": modelo.score(X_train_a, y_train_a),
        "R2_test": modelo.score(X_test_a, y_test_a),
        "RMSE_test": root_mean_squared_error(y_test_a, pred_test),
    })

pd.DataFrame(filas).round(3)


---
## 4. 📋 Chuleta resumen

| Tarea | Código |
|---|---|
| Clasificar | `DecisionTreeClassifier(random_state=42)` |
| Predecir un número | `DecisionTreeRegressor(random_state=42)` |
| Entrenar y predecir | `.fit(X_train, y_train)`, `.predict(X_test)` |
| Accuracy o $R^2$ | `.score(X_test, y_test)` |
| Frenar el árbol | `max_depth=3` o `min_samples_leaf=5` |
| Dibujarlo | `plot_tree(modelo, feature_names=X.columns, max_depth=2)` |
| Train y test, clasificación | `train_test_split(..., stratify=y, random_state=42)` |
| Train y test, regresión | `train_test_split(..., random_state=42)` sin `stratify` |

> 💡 A un árbol no le importa que las variables estén en escalas distintas: solo compara cada una con un umbral. Estandarizar no le cambia el resultado.
